# Aegis Threat Classifier - Graph Neural Network (GNN)
This notebook trains a **Graph Neural Network (GNN)** to classify network packet threats using **PyTorch Geometric (PyG)**.

### Graph Representation of Packets:
1. **Nodes**: Each packet in a batch is treated as a node in a graph.
2. **Node Features**: Concat of embedded payload signatures (extracted via 1D CNN) and scaled metadata features.
3. **Edges**: Constructed on-the-fly using a **K-Nearest Neighbors (KNN)** graph of the metadata space (attributes like TTL, packet length, protocol, and time delta). This links packets with similar traffic characteristics, grouping packet bursts and flows together.
4. **GNN Architecture**: We use **Graph Convolutional Networks (GCN)** to propagate feature context across neighboring packets for threat classification.

## Prerequisite: Install PyTorch Geometric
If you do not have PyTorch Geometric installed, run the cell below to install it based on your PyTorch version.

In [ ]:
# Install PyTorch Geometric (run if not installed)
!pip install torch-geometric

In [ ]:
import os
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# PyTorch Geometric imports
import torch_geometric
from torch_geometric.nn import GCNConv

# Pure PyTorch KNN Graph implementation to bypass external C++ dependencies like pyg-lib
def pytorch_knn_graph(x, k, loop=False):
    dist = torch.cdist(x, x)
    if not loop:
        N = x.size(0)
        # Use fill_diagonal_ to ignore self-connections
        dist.fill_diagonal_(float('inf'))
    _, col = torch.topk(dist, k=k, largest=False, dim=-1)
    row = torch.arange(x.size(0), device=x.device).view(-1, 1).repeat(1, k)
    return torch.stack([col.flatten(), row.flatten()], dim=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch Geometric version: {torch_geometric.__version__}")

## 1. Load Preprocessed Balanced Data

In [ ]:
TRAIN_PATH = 'data/balanced_train.csv'
VAL_PATH = 'data/balanced_val.csv'

def load_dataset(csv_path):
    payload_cols = [f'payload_byte_{i}' for i in range(1, 1501)]
    meta_cols = ['ttl', 'total_len', 't_delta', 'protocol']
    
    dtypes = {f'payload_byte_{i}': np.uint8 for i in range(1, 1501)}
    dtypes['ttl'] = np.uint16
    dtypes['total_len'] = np.uint32
    dtypes['protocol'] = str
    dtypes['t_delta'] = np.float32
    dtypes['label'] = str
    
    chunksize = 100000
    payload_list = []
    meta_list = []
    labels_list = []
    
    for chunk in pd.read_csv(csv_path, dtype=dtypes, chunksize=chunksize):
        payload_list.append(chunk[payload_cols].values)
        meta_list.append(chunk[meta_cols])
        labels_list.append(chunk['label'].values)
        
    payloads = np.concatenate(payload_list, axis=0)
    meta_df = pd.concat(meta_list, ignore_index=True)
    labels = np.concatenate(labels_list, axis=0)
    return payloads, meta_df, labels

print("Loading balanced training set...")
X_train_payload, meta_train_df, y_train_str = load_dataset(TRAIN_PATH)
print("Loading balanced validation set...")
X_val_payload, meta_val_df, y_val_str = load_dataset(VAL_PATH)

## 2. Fit Encoders and Scalers

In [ ]:
# 1. Label Encoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_str)
y_val = label_encoder.transform(y_val_str)
num_classes = len(label_encoder.classes_)

# 2. Protocol Encoder
protocol_encoder = LabelEncoder()
train_protos = meta_train_df['protocol'].astype(str).str.strip().str.lower().fillna('unknown')
val_protos = meta_val_df['protocol'].astype(str).str.strip().str.lower().fillna('unknown')

all_protos = pd.concat([train_protos, val_protos]).unique()
protocol_encoder.fit(all_protos)

meta_train_df['protocol_encoded'] = protocol_encoder.transform(train_protos)
meta_val_df['protocol_encoded'] = protocol_encoder.transform(val_protos)

# 3. Feature Scaling
meta_cols = ['ttl', 'total_len', 't_delta', 'protocol_encoded']
scaler = StandardScaler()
X_train_meta = scaler.fit_transform(meta_train_df[meta_cols].values.astype(np.float32))
X_val_meta = scaler.transform(meta_val_df[meta_cols].values.astype(np.float32))

print(f"Target classes: {list(label_encoder.classes_)}")

del meta_train_df, meta_val_df, train_protos, val_protos
gc.collect()

## 3. Prepare PyTorch DataLoaders

In [ ]:
train_dataset = TensorDataset(
    torch.tensor(X_train_payload, dtype=torch.uint8),
    torch.tensor(X_train_meta, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(X_val_payload, dtype=torch.uint8),
    torch.tensor(X_val_meta, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
print(f"Batches in Train: {len(train_loader)} | Val: {len(val_loader)}")

## 4. Define GCN Model & Focal Loss
Within the GNN forward pass, we:
1. Embed the payload bytes and pass them through a 1D Convolution + Pool to extract signature embeddings.
2. Concatenate with the scaled metadata attributes.
3. Build a **KNN Graph** (`pytorch_knn_graph`) based on the metadata attributes to connect packets within similar flows.
4. Propagate packet representations using GCN layers to classify threats.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        return focal_loss.mean()

class PayloadGCN(nn.Module):
    def __init__(self, num_classes=6, k_neighbors=5):
        super(PayloadGCN, self).__init__()
        self.k_neighbors = k_neighbors
        
        # Node Feature Extractor (CNN)
        self.embedding = nn.Embedding(256, 32)
        self.conv1 = nn.Conv1d(32, 64, kernel_size=7, stride=4, padding=3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2, 2)
        
        # CNN flattens to 64 * 187 = 11,968 dimensions
        self.project_payload = nn.Sequential(
            nn.Linear(64 * 187, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Combined input size: 128 (payload embedding) + 4 (metadata)
        self.gcn1 = GCNConv(128 + 4, 64)
        self.gcn2 = GCNConv(64, 64)
        
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, payload, metadata):
        # 1. Node Feature Extraction
        x_embed = self.embedding(payload.long()).transpose(1, 2)  # (nodes, 32, 1500)
        x_conv = self.pool(self.relu(self.conv1(x_embed)))         # (nodes, 64, 187)
        x_flat = x_conv.flatten(1)                                # (nodes, 11968)
        payload_feats = self.project_payload(x_flat)              # (nodes, 128)

        # Node features (concat payload embedding and metadata)
        h = torch.cat((payload_feats, metadata), dim=1)           # (nodes, 132)
        
        # 2. Graph Edges Construction via KNN (custom PyTorch implementation to avoid pyg-lib dependency)
        edge_index = pytorch_knn_graph(metadata, k=self.k_neighbors, loop=False)
        
        # 3. GNN Message Passing (Convolutional Layers)
        h_graph = F.relu(self.gcn1(h, edge_index))
        h_graph = F.dropout(h_graph, p=0.2, training=self.training)
        h_graph = F.relu(self.gcn2(h_graph, edge_index))
        
        # 4. Classification Output
        out = self.fc(h_graph)
        return out

## 5. Train and Save GNN Model

In [ ]:
model = PayloadGCN(num_classes=num_classes, k_neighbors=5).to(device)
criterion = FocalLoss(gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Setup model saving directory
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)
best_val_acc = 0.0

EPOCHS = 5
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting Graph Neural Network (GCN) Training...")
for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for payloads, metas, labels in train_loader:
        payloads, metas, labels = payloads.to(device), metas.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(payloads, metas)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * payloads.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for payloads, metas, labels in val_loader:
            payloads, metas, labels = payloads.to(device), metas.to(device), labels.to(device)
            outputs = model(payloads, metas)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * payloads.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    t_loss = train_loss / train_total
    t_acc = train_correct / train_total
    v_loss = val_loss / val_total
    v_acc = val_correct / val_total
    
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.2%} | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.2%}")
    
    # Save model weights to your PC when accuracy increases
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        save_path = os.path.join(MODEL_DIR, 'gnn_model.pth')
        torch.save(model.state_dict(), save_path)
        print(f"  --> Best GNN model weights saved to: {os.path.abspath(save_path)}")
    
print("GNN training complete!")

## 6. Plot Learning History

In [ ]:
epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_acc'], label='Train Accuracy')
plt.plot(epochs_range, history['val_acc'], label='Validation Accuracy')
plt.title('GNN Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_loss'], label='Train Loss')
plt.plot(epochs_range, history['val_loss'], label='Validation Loss')
plt.title('GNN Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss (Focal Loss)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 7. Evaluate GNN Performance

In [ ]:
# Load best saved GNN model to verify performance
best_save_path = os.path.join(MODEL_DIR, 'gnn_model.pth')
if os.path.exists(best_save_path):
    model.load_state_dict(torch.load(best_save_path, map_location=device, weights_only=True))
    print("Best saved GNN weights loaded for evaluation.")

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for payloads, metas, labels in val_loader:
        payloads, metas = payloads.to(device), metas.to(device)
        outputs = model(payloads, metas)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.numpy())

print("\n=========================================")
print(" GNN Evaluation: Classification Report")
print("=========================================")
print(classification_report(all_targets, all_preds, target_names=label_encoder.classes_, zero_division=0))

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title("GNN Confusion Matrix", fontsize=13)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()